In [33]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# changeed animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# updated with your username and password and CRUD Python module name

username = "aacuser"
password = "Joyb0y444"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#Added Grazioso Salvare's Logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    # Grazioso Salvare logo with hyperlink
    html.Center(
        html.A(
            html.Img(
                src='data:image/png;base64,{}'.format(
                    encoded_image.decode()
                ),
                style={'height': '120px'}
            ),
            href='https://www.snhu.edu',
            target='_blank'
        )
    ),

    # Dashboard creator identification
    html.Center(
        html.P('Dashboard Created by Robert Cardoza')
    ),
    
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Hr(),
    html.Div(
        
        # Interactive filtering options for rescue dog categories
        html.Div([
            html.H3("Select Rescue Type"),
            dcc.RadioItems(
                id='filter-type',
                options=[
                    {'label': 'Water Rescue', 'value': 'water'},
                    {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                    {'label': 'Disaster or Individual Tracking', 'value': 'disaster'},
                    {'label': 'Reset (Show All Animals)', 'value': 'reset'}
                ],
                value='reset',
                labelStyle={'display': 'block'},
                style={'margin': '10px'}
            )
        ]),
        
    ),
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {
                "name": i,
                "id": i,
                "deletable": False,
                "selectable": True
            }
            for i in df.columns
        ],
        data=df.to_dict('records'),

        # Enable sorting
        sort_action='native',
        
        # Enable filtering
        filter_action='native',

        # Enable pagination
        page_action='native',
        page_current=0,
        page_size=10,

        # Allow selecting a single row for the map
        row_selectable='single',
        selected_rows=[],

        # Improve table appearance
        style_table={
        'overflowX': 'auto'
        },
        style_cell={
            'textAlign': 'left',
            'padding': '10px',
            'minWidth': '100px',
            'maxWidth': '200px',
            'whiteSpace': 'normal'
        },
        style_header={
            'backgroundColor': 'lightgrey',
            'fontWeight': 'bold'
        }

    ),

    html.Br(),
    html.Hr(),
    #This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    # Water Rescue
    if filter_type == 'water':
        query = {
            'animal_type': 'Dog',
            'breed': {
                '$in': [
                    'Labrador Retriever Mix',
                    'Chesapeake Bay Retriever',
                    'Newfoundland'
                ]
            },
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {
                '$gte': 26,
                '$lte': 156
            }
        }
        
    # Mountain or Wilderness Rescue
    elif filter_type == 'mountain':
        query = {
            'animal_type': 'Dog',
            'breed': {
                '$in': [
                    'German Shepherd',
                    'Alaskan Malamute',
                    'Old English Sheepdog',
                    'Siberian Husky',
                    'Rottweiler'
                ]
            },
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {
                '$gte': 26,
                '$lte': 156
            }
        }

    # Disaster or Individual Tracking
    elif filter_type == 'disaster':
        query = {
            'animal_type': 'Dog',
            'breed': {
                '$in': [
                    'Doberman Pinscher',
                    'German Shepherd',
                    'Golden Retriever',
                    'Bloodhound',
                    'Rottweiler'
                ]
            },
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {
                '$gte': 20,
                '$lte': 300
            }
        }
        
    # Reset: Return all animals
    else:
        query = {}
        
    # Retrieve matching records using the CRUD Python module
    filtered_data = pd.DataFrame.from_records(db.read(query))
    
    # Remove MongoDB's ObjectId column
    if '_id' in filtered_data.columns:
        filtered_data.drop(columns=['_id'], inplace=True)
    return filtered_data.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):

    # Check whether the table contains data
    if viewData is None or len(viewData) == 0:
        return []
    
    # Convert filtered records into a DataFrame
    dff = pd.DataFrame.from_dict(viewData)
    
    # Count animals by breed
    breed_counts = dff['breed'].value_counts()
    
    # Select the 10 most common breeds
    top_breeds = breed_counts.head(10)
    
    # Combine remaining breeds into Other
    other_count = breed_counts.iloc[10:].sum()
    if other_count > 0:
        top_breeds['Other'] = other_count
        
    # Convert counts into a DataFrame
    chart_data = top_breeds.reset_index()
    chart_data.columns = ['breed', 'count']
    
    # Create the pie chart
    fig = px.pie(
        chart_data,
        names='breed',
        values='count',
        title='Distribution of Animal Breeds'
    )
    
    # Configure chart appearance
    fig.update_layout(
        height=500,
        margin=dict(l=20, r=20, t=60, b=20),
        legend=dict(
            orientation='v',
            yanchor='top',
            y=1,
            xanchor='left',
            x=1,
            font=dict(size=11)
        )
    )
    return [
        dcc.Graph(
            figure=fig,
            style={'width': '100%'}
        )
    ]

#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)

def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': {'column_id': i},
        'backgroundColor': '#D2F3FF'
    } for i in selected_columns]

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])

def update_map(viewData, index):
    if viewData is None or len(viewData) == 0:
        return []
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None or len(index) == 0:
        row = 0    
    else:
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://genevaenigma-newtontoast-3000.codio.io/proxy/8050/
